<a href="https://colab.research.google.com/github/melatisurya/240441100142_Melati_Surya_Ayu_Nitiyas_Tugas2/blob/main/240441100142_Melati_Surya_Ayu_Nitiyas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Menggunakan device:", device)

Menggunakan device: cpu


2. Dataset CIFAR 10

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

val_size = int(0.1 * len(full_train))
train_size = len(full_train) - val_size
train_set, val_set = torch.utils.data.random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')
print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

Train: 45000, Val: 5000, Test: 10000


3. Model CNN sederhana

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3,32,3,padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2,2))
        self.block2 = nn.Sequential(nn.Conv2d(32,64,3,padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2,2))
        self.block3 = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2,2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)  # softmax implisit via CrossEntropyLoss
        )
    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        return self.classifier(x)

4. Model B: Mini-ResNet (Residual Block Buatan Sendiri)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity   # residual connection
        return self.relu(out)

class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3,32,3,padding=1,bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.stage1 = nn.Sequential(ResidualBlock(32,32,1), ResidualBlock(32,32,1))
        self.stage2 = nn.Sequential(ResidualBlock(32,64,2), ResidualBlock(64,64,1))
        self.stage3 = nn.Sequential(ResidualBlock(64,128,2), ResidualBlock(128,128,1))
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.stem(x); x = self.stage1(x); x = self.stage2(x); x = self.stage3(x)
        x = self.pool(x); x = torch.flatten(x,1)
        return self.fc(x)

5. Fungsi Training & Evaluasi

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss/total, correct/total

def train_model(model, train_loader, val_loader, epochs, lr, model_name="Model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss, train_acc = running_loss/total, correct/total
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | Train Loss:{train_loss:.4f} Acc:{train_acc:.4f} | Val Loss:{val_loss:.4f} Acc:{val_acc:.4f}")

    total_time = time.time() - start_time
    print(f"[{model_name}] Total waktu training: {total_time:.2f} detik ({total_time/60:.2f} menit)")
    return history, total_time

6. Training Kedua Model (Hyperparameter Identik)

In [ ]:
EPOCHS = 15
LEARNING_RATE = 0.001

model_A = SimpleCNN(num_classes=10)
model_B = MiniResNet(num_classes=10)

print("Jumlah parameter Model A:", count_parameters(model_A))
print("Jumlah parameter Model B:", count_parameters(model_B))

print("\n--- Training Model A ---")
history_A, time_A = train_model(model_A, train_loader, val_loader, EPOCHS, LEARNING_RATE, "Model A")

print("\n--- Training Model B ---")
history_B, time_B = train_model(model_B, train_loader, val_loader, EPOCHS, LEARNING_RATE, "Model B")

Jumlah parameter Model A: 620362
Jumlah parameter Model B: 696618

--- Training Model A ---
[Model A] Epoch 1/15 | Train Loss:1.7021 Acc:0.3652 | Val Loss:1.4897 Acc:0.4554
[Model A] Epoch 2/15 | Train Loss:1.3760 Acc:0.4995 | Val Loss:1.2394 Acc:0.5580
[Model A] Epoch 3/15 | Train Loss:1.2187 Acc:0.5638 | Val Loss:1.0802 Acc:0.6156
[Model A] Epoch 4/15 | Train Loss:1.0977 Acc:0.6092 | Val Loss:0.9769 Acc:0.6476
[Model A] Epoch 5/15 | Train Loss:1.0267 Acc:0.6372 | Val Loss:0.9360 Acc:0.6664
[Model A] Epoch 6/15 | Train Loss:0.9724 Acc:0.6582 | Val Loss:0.8794 Acc:0.6792
[Model A] Epoch 7/15 | Train Loss:0.9218 Acc:0.6783 | Val Loss:0.8218 Acc:0.7080
[Model A] Epoch 8/15 | Train Loss:0.8816 Acc:0.6908 | Val Loss:0.8257 Acc:0.7058
[Model A] Epoch 9/15 | Train Loss:0.8527 Acc:0.7017 | Val Loss:0.7994 Acc:0.7166
[Model A] Epoch 10/15 | Train Loss:0.8243 Acc:0.7144 | Val Loss:0.7679 Acc:0.7276
[Model A] Epoch 11/15 | Train Loss:0.7979 Acc:0.7271 | Val Loss:0.7433 Acc:0.7380
[Model A] Epoch

7. Evaluasi Test Set

In [ ]:
criterion = nn.CrossEntropyLoss()
test_loss_A, test_acc_A = evaluate(model_A, test_loader, criterion)
test_loss_B, test_acc_B = evaluate(model_B, test_loader, criterion)

print(f"Model A - Test Loss: {test_loss_A:.4f} | Test Accuracy: {test_acc_A:.4f}")
print(f"Model B - Test Loss: {test_loss_B:.4f} | Test Accuracy: {test_acc_B:.4f}")

8. kode grafik

In [ ]:
EPOCHS_RANGE = range(1, len(history_A["train_loss"]) + 1)

# ============================================================
# 1. GRAFIK LOSS (Training vs Validation, kedua model)
# ============================================================
plt.figure(figsize=(8,5))
plt.plot(EPOCHS_RANGE, history_A["train_loss"], 'o-', label="Model A - Train Loss")
plt.plot(EPOCHS_RANGE, history_A["val_loss"], 'o--', label="Model A - Val Loss")
plt.plot(EPOCHS_RANGE, history_B["train_loss"], 's-', label="Model B - Train Loss")
plt.plot(EPOCHS_RANGE, history_B["val_loss"], 's--', label="Model B - Val Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("grafik_loss.png", dpi=150)
plt.show()

# ============================================================
# 2. GRAFIK ACCURACY (Training vs Validation, kedua model)
# ============================================================
plt.figure(figsize=(8,5))
plt.plot(EPOCHS_RANGE, history_A["train_acc"], 'o-', label="Model A - Train Acc")
plt.plot(EPOCHS_RANGE, history_A["val_acc"], 'o--', label="Model A - Val Acc")
plt.plot(EPOCHS_RANGE, history_B["train_acc"], 's-', label="Model B - Train Acc")
plt.plot(EPOCHS_RANGE, history_B["val_acc"], 's--', label="Model B - Val Acc")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("grafik_accuracy.png", dpi=150)
plt.show()

# ============================================================
# 3. CONFUSION MATRIX MODEL A & B (berdampingan)
# ============================================================
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

labels_A, preds_A = get_predictions(model_A, test_loader)
labels_B, preds_B = get_predictions(model_B, test_loader)
cm_A = confusion_matrix(labels_A, preds_A)
cm_B = confusion_matrix(labels_B, preds_B)

fig, axes = plt.subplots(1, 2, figsize=(16,7))
ConfusionMatrixDisplay(cm_A, display_labels=classes).plot(ax=axes[0], cmap="Blues", xticks_rotation=45, colorbar=False)
axes[0].set_title("Confusion Matrix - Model A (Simple CNN)")
ConfusionMatrixDisplay(cm_B, display_labels=classes).plot(ax=axes[1], cmap="Greens", xticks_rotation=45, colorbar=False)
axes[1].set_title("Confusion Matrix - Model B (Mini-ResNet)")
plt.tight_layout()
plt.savefig("confusion_matrix_comparison.png", dpi=150)
plt.show()

# ============================================================
# 4. BAR CHART: PARAMETER, WAKTU TRAINING, TEST ACCURACY
# ============================================================
param_A, param_B = count_parameters(model_A), count_parameters(model_B)

fig, axes = plt.subplots(1, 3, figsize=(16,5))

axes[0].bar(["Model A","Model B"], [param_A, param_B], color=["#4C72B0","#55A868"])
axes[0].set_title("Jumlah Parameter")
for i,v in enumerate([param_A, param_B]):
    axes[0].text(i, v, f"{v:,}", ha='center', va='bottom', fontsize=9)

axes[1].bar(["Model A","Model B"], [time_A, time_B], color=["#4C72B0","#55A868"])
axes[1].set_title("Waktu Training (detik)")
for i,v in enumerate([time_A, time_B]):
    axes[1].text(i, v, f"{v:.1f}s", ha='center', va='bottom', fontsize=9)

axes[2].bar(["Model A","Model B"], [test_acc_A, test_acc_B], color=["#4C72B0","#55A868"])
axes[2].set_title("Test Accuracy")
axes[2].set_ylim(0,1)
for i,v in enumerate([test_acc_A, test_acc_B]):
    axes[2].text(i, v, f"{v:.4f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("perbandingan_bar_chart.png", dpi=150)
plt.show()

# ============================================================
# 5. TABEL RINGKASAN
# ============================================================
summary = pd.DataFrame({
    "Model": ["Model A (Simple CNN)", "Model B (Mini-ResNet)"],
    "Jumlah Parameter": [param_A, param_B],
    "Waktu Training (detik)": [round(time_A,2), round(time_B,2)],
    "Waktu Training (menit)": [round(time_A/60,2), round(time_B/60,2)],
    "Test Loss": [round(test_loss_A,4), round(test_loss_B,4)],
    "Test Accuracy": [round(test_acc_A,4), round(test_acc_B,4)],
})
print(summary.to_string(index=False))
summary.to_csv("ringkasan_perbandingan.csv", index=False)

Seed & device
torch.manual_seed(42) mengunci angka acak supaya hasil training bisa direproduksi — kalau notebook dijalankan ulang, hasilnya akan sama persis. Device otomatis pakai GPU (cuda) kalau tersedia, kalau tidak fallback ke CPU.

Transform / augmentasi data

Untuk data training: RandomCrop dan RandomHorizontalFlip menambah variasi gambar secara acak (data augmentation) supaya model tidak hafal data mentah-mentah, ini mengurangi overfitting.
Untuk data test/validation: tidak ada augmentasi, hanya normalisasi — karena kita ingin mengevaluasi model pada data apa adanya.
Angka normalisasi (0.4914, 0.4822, 0.4465) dan (0.2470, 0.2435, 0.2616) adalah mean dan std dari dataset CIFAR-10 (angka standar yaang dibiasa dipakai).Split data
Data training asli (50.000 gambar) dipecah jadi 90% training dan 10% validation. Validation dipakai untuk memantau performa model selama training (bukan untuk melatih), sedangkan test set (10.000 gambar terpisah) hanya dipakai di akhir untuk skor final yang objektif.

2. Model A — CNN Sederhana

Alurnya: Conv → ReLU → MaxPool, diulang 3 kali, lalu masuk ke Fully Connected layer.

Conv2d: mendeteksi fitur visual (tepi, tekstur, pola) dari gambar.
ReLU: fungsi aktivasi non-linear, membuat model bisa belajar pola kompleks (bukan cuma garis lurus).
MaxPool: memperkecil ukuran gambar (32→16→8→4) sambil mempertahankan fitur penting, juga mengurangi jumlah parameter.
Flatten + Linear: mengubah fitur gambar (dalam bentuk grid) jadi vektor, lalu diklasifikasikan ke 10 kelas.
Dropout(0.5): mematikan 50% neuron secara acak saat training untuk mencegah overfitting.
Tidak ada Softmax eksplisit di akhir karena nn.CrossEntropyLoss di PyTorch sudah otomatis menerapkan softmax secara internal.

Ini model paling dasar — tanpa BatchNorm, tanpa skip connection.


3. Model B — Mini-ResNet (Residual Connection)

Inti dari residual block:

python
out = self.relu(self.bn1(self.conv1(x)))
out = self.bn2(self.conv2(out))
out += identity   # <-- ini bagian residual/skip connection
out = self.relu(out)

Idenya: alih-alih memaksa layer mempelajari transformasi penuh H(x), residual block hanya mempelajari selisihnya F(x) = H(x) - x, lalu ditambahkan kembali ke input asli (identity). Ini disebut skip connection.
Kenapa ini penting?

Memudahkan gradient mengalir ke layer-layer awal saat backpropagation (mengatasi masalah vanishing gradient) sehingga jaringan yang dalam tetap bisa dilatih dengan baik.
Kalau suatu blok tidak berguna, model bisa "melewatinya" dengan membuat F(x) ≈ 0, sehingga tidak merusak performa.

Shortcut/identity path:

python
if stride != 1 or in_channels != out_channels:
    self.shortcut = nn.Sequential(Conv2d 1x1, BatchNorm)

Kalau dimensi channel atau ukuran spasial berubah (misal 32→64 channel atau downsampling), skip connection biasa tidak bisa langsung dijumlahkan (dimensi tidak cocok). Maka dipakai konvolusi 1×1 untuk menyesuaikan dimensi identity agar bisa dijumlahkan dengan out.

Struktur MiniResNet:

stem: layer awal untuk ekstraksi fitur dasar.
stage1, stage2, stage3: masing-masing berisi 2 residual block. Stage2 dan stage3 melakukan downsampling (stride=2) sambil menambah jumlah channel (32→64→128), sama seperti pola MaxPool di Model A tapi dengan residual.
AdaptiveAvgPool2d((1,1)): meratakan feature map jadi 1×1 per channel (Global Average Pooling) — lebih efisien parameter dibanding Flatten+Linear besar.

4. Fungsi Training & Evaluasi (Cell 5)
evaluate(): menjalankan model dalam mode eval() (mematikan dropout/batchnorm training behavior), tanpa menghitung gradient (torch.no_grad()) agar lebih cepat dan hemat memori. Fungsi ini dipakai baik untuk validation maupun test.
train_model(): loop utama training. Untuk setiap epoch:
Mode train() diaktifkan.
Forward pass → hitung loss → backward() (hitung gradient) → optimizer.step() (update bobot).
Setelah semua batch selesai, dihitung train_loss/train_acc, lalu dievaluasi ke val_loader untuk dapat val_loss/val_acc.
Semua metrik disimpan ke history per epoch — inilah yang nanti dipakai untuk membuat grafik.
time.time() di awal dan akhir dipakai mengukur total waktu training, salah satu metrik wajib di tugas.

5. Kenapa Perbandingan "Adil"?

Model A dan Model B dilatih dengan hyperparameter identik: epochs=15, batch_size=128, optimizer Adam, lr=0.001, dataset dan split yang sama persis. Jadi perbedaan hasil murni berasal dari perbedaan arsitektur, bukan karena satu model "diuntungkan" settingan training yang lebih baik.

6. Penjelasan Grafik (Cell 8)
Grafik Loss: Kalau val_loss mulai naik sementara train_loss terus turun → tanda overfitting. Biasanya Model A (tanpa BatchNorm/residual) lebih rentan overfitting dibanding Model B.
Grafik Accuracy: Semakin cepat kurva val_acc naik dan stabil di angka tinggi, semakin baik model belajar generalisasi.
Confusion Matrix: Diagonal = prediksi benar. Nilai di luar diagonal menunjukkan kelas mana yang sering tertukar (contoh klasik di CIFAR-10: cat vs dog, atau deer vs horse karena bentuk visual mirip).
Bar chart parameter/waktu/akurasi: Memvisualisasikan trade-off — biasanya Model B (residual) punya akurasi lebih tinggi tapi butuh waktu training sedikit lebih lama karena BatchNorm dan struktur lebih dalam; sementara jumlah parameter bisa jadi mirip atau bahkan lebih sedikit tergantung desain channel yang dipakai di sini (32→64→128, sama di kedua model).